In [26]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt  # for making figures
%matplotlib inline

In [27]:
# Read in all the words
words = open('../names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [28]:
len(words)

32033

In [29]:
# Build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [30]:
# build the dataset

block_size = 3  # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:

    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]  # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)


emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [31]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [32]:
C = torch.randn((27, 2))

In [33]:
C[X]

tensor([[[ 0.0474, -0.0699],
         [ 0.0474, -0.0699],
         [ 0.0474, -0.0699]],

        [[ 0.0474, -0.0699],
         [ 0.0474, -0.0699],
         [-0.9384, -2.0842]],

        [[ 0.0474, -0.0699],
         [-0.9384, -2.0842],
         [-0.9094, -0.4697]],

        [[-0.9384, -2.0842],
         [-0.9094, -0.4697],
         [-0.9094, -0.4697]],

        [[-0.9094, -0.4697],
         [-0.9094, -0.4697],
         [ 0.5595,  0.2537]],

        [[ 0.0474, -0.0699],
         [ 0.0474, -0.0699],
         [ 0.0474, -0.0699]],

        [[ 0.0474, -0.0699],
         [ 0.0474, -0.0699],
         [ 2.3720, -1.4338]],

        [[ 0.0474, -0.0699],
         [ 2.3720, -1.4338],
         [ 0.7141,  0.7896]],

        [[ 2.3720, -1.4338],
         [ 0.7141,  0.7896],
         [-1.7148, -1.1266]],

        [[ 0.7141,  0.7896],
         [-1.7148, -1.1266],
         [ 1.4610,  0.0802]],

        [[-1.7148, -1.1266],
         [ 1.4610,  0.0802],
         [-1.7148, -1.1266]],

        [[ 1.4610,  0

In [37]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [39]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [40]:
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1).shape

torch.Size([32, 6])

In [41]:
torch.cat(torch.unbind(emb, 1), 1).shape

torch.Size([32, 6])

In [42]:
emb.shape

torch.Size([32, 3, 2])

In [43]:
emb.view(32, 6)

tensor([[ 0.0474, -0.0699,  0.0474, -0.0699,  0.0474, -0.0699],
        [ 0.0474, -0.0699,  0.0474, -0.0699, -0.9384, -2.0842],
        [ 0.0474, -0.0699, -0.9384, -2.0842, -0.9094, -0.4697],
        [-0.9384, -2.0842, -0.9094, -0.4697, -0.9094, -0.4697],
        [-0.9094, -0.4697, -0.9094, -0.4697,  0.5595,  0.2537],
        [ 0.0474, -0.0699,  0.0474, -0.0699,  0.0474, -0.0699],
        [ 0.0474, -0.0699,  0.0474, -0.0699,  2.3720, -1.4338],
        [ 0.0474, -0.0699,  2.3720, -1.4338,  0.7141,  0.7896],
        [ 2.3720, -1.4338,  0.7141,  0.7896, -1.7148, -1.1266],
        [ 0.7141,  0.7896, -1.7148, -1.1266,  1.4610,  0.0802],
        [-1.7148, -1.1266,  1.4610,  0.0802, -1.7148, -1.1266],
        [ 1.4610,  0.0802, -1.7148, -1.1266,  0.5595,  0.2537],
        [ 0.0474, -0.0699,  0.0474, -0.0699,  0.0474, -0.0699],
        [ 0.0474, -0.0699,  0.0474, -0.0699,  0.5595,  0.2537],
        [ 0.0474, -0.0699,  0.5595,  0.2537,  1.4610,  0.0802],
        [ 0.5595,  0.2537,  1.4610,  0.0